In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

In [ ]:
from marine_qc import (
    combine_qc_results,
    do_climatology_check,
    do_datetime_check,
    do_hard_limit_check,
    do_maritime_check,
    do_multiple_individual_check,
    do_position_check,
    do_valid_value_check,
    do_wind_consistency_check,
    plot_latitude_longitude,
)
from marine_qc.helpers.external_clim import get_climatological_value

In [ ]:
from data import get_climatology_data, get_individual_data

# How to use quality control checks with individual reports

We need some text!!!

In [ ]:
data = get_individual_data()
data

In [ ]:
qc_pos = do_position_check(
    lat=data.lat,
    lon=data.lon,
)
pd.DataFrame({"location": data.location, "lat": data.lat, "lon": data.lon, "qc_pos": qc_pos})

In [ ]:
qc_datetime = do_datetime_check(
    data.date,
)
pd.DataFrame({"location": data.location, "date": data.date, "qc_datetime": qc_datetime})

In [ ]:
qc_valid = do_valid_value_check(
    data.sea_surface_temperature,
)
pd.DataFrame({"location": data.location, "sea_surface_temperature": data.sea_surface_temperature, "qc_valid": qc_valid})

In [ ]:
qc_hard = do_hard_limit_check(
    data.sea_surface_temperature,
    limits=(10, 30),
)
pd.DataFrame({"location": data.location, "sea_surface_temperature": data.sea_surface_temperature, "qc_hard": qc_hard})

In [ ]:
qc_wind = do_wind_consistency_check(
    wind_speed=data.wind_speed,
    wind_direction=data.wind_direction,
)
pd.DataFrame({"location": data.location, "wind_speed": data.wind_speed, "wind_direction": data.wind_direction, "qc_wind": qc_wind})

In [ ]:
climatology_data = get_climatology_data()
climatology_data

In [ ]:
climatology_data.land_sea_mask.plot()

In [ ]:
qc_sea = do_maritime_check(
    lat=data.lat,
    lon=data.lon,
    sea_land_mask=climatology_data.land_sea_mask,
    sea_flag=0,
)
pd.DataFrame({"location": data.location, "lat": data.lat, "lon": data.lon, "qc_sea": qc_sea})

In [ ]:
climatology_data.sst.plot()

In [ ]:
clim_sst = get_climatological_value(climatology_data.sst, lat=data.lat, lon=data.lon)

In [ ]:
qc_clim = do_climatology_check(
    value=data.sea_surface_temperature,
    climatology=climatology_data.sst,
    maximum_anomaly=5.0,
    lat=data.lat,
    lon=data.lon,
)
pd.DataFrame(
    {
        "location": data.location,
        "lat": data.lat,
        "lon": data.lon,
        "sea_surface_temperature": data.sea_surface_temperature,
        "climatology": clim_sst,
        "qc_clim": qc_clim,
    }
)

In [ ]:
qc_dict = {
    "positional_check": {
        "func": "do_position_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
        },
    },
    "datetime_check": {
        "func": "do_datetime_check",
        "names": {"date": "date"},
    },
    "hard_limit_check": {"func": "do_hard_limit_check", "names": {"value": "sea_surface_temperature"}, "arguments": {"limits": (10, 29)}},
    "maritime_check": {
        "func": "do_maritime_check",
        "names": {
            "lat": "lat",
            "lon": "lon",
        },
        "arguments": {
            "sea_land_mask": climatology_data.land_sea_mask,
            "sea_flag": 0,
        },
    },
    "climatology_check": {
        "func": "do_climatology_check",
        "names": {"value": "sea_surface_temperature"},
        "arguments": {
            "climatology": climatology_data.sst,
            "maximum_anomaly": 5.0,
            "lat": data.lat,
            "lon": data.lon,
        },
    },
    "wind_consistency_check": {
        "func": "do_wind_consistency_check",
        "names": {"wind_speed": "wind_speed", "wind_direction": "wind_direction"},
    },
}

In [ ]:
qc_multi = do_multiple_individual_check(
    data,
    qc_dict,
    return_method="failed",
)
qc_multi

In [ ]:
qc_flag = combine_qc_results(qc_multi)
pd.DataFrame({**data, "qc_flag": qc_flag})

In [ ]:
plot = plot_latitude_longitude(data.lat, data.lon, qc_flag, marker_size=15, add_coastlines=True)